# 35 CPU evaluation, fusion, and reports

CPU。player-season、event-method projection、fusion、手法比較、研究用 tables/figures/video links、HTML report をまとめます。

各 stage は `BASE_DIR/reports/preflight/compact_runs/` に JSONL log を追記します。既に期待 artifact が揃っている stage は skip します。再実行したい場合は `FORCE_RERUN_ALL=True` にしてください。

In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import shutil
import sys


def _is_transport_endpoint_error(exc) -> bool:
    return getattr(exc, 'errno', None) == 107 or 'Transport endpoint is not connected' in str(exc)


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    try:
        already_mounted = (mountpoint / 'MyDrive').exists()
    except OSError as exc:
        if not _is_transport_endpoint_error(exc):
            raise
        print('Drive mount looks stale (Transport endpoint is not connected). Trying force_remount.')
        try:
            drive.mount(str(mountpoint), force_remount=True)
            return
        except Exception as remount_exc:
            raise RuntimeError(
                'Colab Drive mount is stale. Restart the runtime, mount Drive again, then rerun 35. '
                'Completed stages are resumable; no artifact deletion is required.'
            ) from remount_exc
    if already_mounted:
        print('Drive already mounted at /content/drive')
        return
    try:
        drive.mount(str(mountpoint))
    except ValueError as exc:
        message = str(exc)
        if 'Mountpoint must not already contain files' in message or 'already mounted' in message:
            print(f'Drive mount warning: {message}')
            try:
                needs_remount = not (mountpoint / 'MyDrive').exists()
            except OSError as exists_exc:
                needs_remount = _is_transport_endpoint_error(exists_exc)
            if needs_remount:
                drive.mount(str(mountpoint), force_remount=True)
        else:
            raise
    except Exception as exc:
        print(f'Colab Drive mount skipped or failed: {exc}')


# v2 をデフォルトにします。v1 を再実行したい時だけここを書き換えてください。
os.environ.setdefault('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
os.environ.setdefault('BATTING_CODE_ROOT', '/content/drive/MyDrive/codex/batting_codex_handoff')


def _is_repo_dir(path: Path) -> bool:
    return (
        (path / 'src' / 'sport_pipeline' / '__init__.py').exists()
        and (path / 'configs').exists()
        and (path / 'notebooks').exists()
    )


def _resolve_repo_dir() -> Path:
    fixed = Path(os.environ['BATTING_CODE_ROOT'])
    mydrive = Path('/content/drive/MyDrive')
    candidates = [
        fixed,
        Path('/content/drive/MyDrive/codex/batting_codex_handoff'),
        Path('/content/drive/MyDrive/batting_codex_handoff'),
        Path.cwd(),
        *Path.cwd().parents,
    ]
    checked = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            return candidate
    raise FileNotFoundError(
        'sport_pipeline repo が見つかりません。Drive mount と repo 配置を確認してください。\n'
        f'BATTING_CODE_ROOT={fixed}\n'
        '推奨配置: /content/drive/MyDrive/codex/batting_codex_handoff\n'
        '別配置の場合は compact notebook の最初の cell より前に次を実行してください。\n'
        '  %env BATTING_CODE_ROOT=/content/drive/MyDrive/あなたの配置/batting_codex_handoff\n'
        f'MyDrive exists={mydrive.exists()}\n'
        '確認した候補:\n- ' + '\n- '.join(checked)
    )


def _use_local_code_cache() -> bool:
    return os.environ.get('BASEBALL_VISION_USE_CODE_CACHE', '1').strip().lower() not in {'0', 'false', 'no'}


def _copy_repo_to_local_cache(repo_dir: Path) -> Path:
    if not _use_local_code_cache():
        print('Code cache disabled; using repo directly:', repo_dir)
        return repo_dir
    if not str(repo_dir).startswith('/content/drive/'):
        print('Code repo is already local:', repo_dir)
        return repo_dir

    cache_repo = Path(os.environ.get(
        'BATTING_CODE_CACHE_ROOT',
        '/content/cache/baseball_vision/code/batting_codex_handoff',
    ))
    if cache_repo.exists():
        shutil.rmtree(cache_repo)
    cache_repo.parent.mkdir(parents=True, exist_ok=True)
    ignore = shutil.ignore_patterns(
        '.git',
        '__pycache__',
        '*.pyc',
        '.ipynb_checkpoints',
        '.pytest_cache',
        'mlruns',
        'wandb',
    )
    shutil.copytree(repo_dir, cache_repo, ignore=ignore)
    if not _is_repo_dir(cache_repo):
        raise RuntimeError(f'ローカル code cache に repo 構造をコピーできませんでした: {cache_repo}')
    if not (cache_repo / 'src' / 'sport_pipeline' / 'reports').exists():
        raise RuntimeError(f'ローカル code cache に sport_pipeline.reports がありません: {cache_repo}')
    print('Copied code repo to local cache:', cache_repo)
    return cache_repo


def _prefer_repo_src(repo_dir: Path) -> Path:
    src_dir = repo_dir / 'src'
    for old in list(sys.path):
        if old != str(src_dir) and old.endswith('/src') and 'batting_codex_handoff/src' in old:
            sys.path.remove(old)
    if str(src_dir) not in sys.path:
        sys.path.insert(0, str(src_dir))

    if str(repo_dir).startswith('/content/cache/'):
        removed = []
        for name, module in list(sys.modules.items()):
            if name == 'sport_pipeline' or name.startswith('sport_pipeline.'):
                module_file = str(getattr(module, '__file__', '') or '')
                if '/content/drive/' in module_file:
                    del sys.modules[name]
                    removed.append(name)
        if removed:
            print('Dropped Drive-backed cached sport_pipeline modules:', len(removed))
    return src_dir


_mount_drive_if_needed()
DRIVE_REPO_DIR = _resolve_repo_dir()
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
REPO_DIR = _copy_repo_to_local_cache(DRIVE_REPO_DIR)
os.environ['BATTING_CODE_ROOT'] = str(REPO_DIR)
NOTEBOOK_DIR = REPO_DIR / 'notebooks'
RUN_PROFILE_NAME = os.environ['BASEBALL_VISION_RUN_PROFILE']
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME
src_dir = _prefer_repo_src(REPO_DIR)
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(f'sport_pipeline を import できません: {src_dir}')

from sport_pipeline.compact_notebooks import CompactRunLogger
from sport_pipeline.pipeline.run_profile import artifact_id, load_run_profile, run_id

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
FULL_RUN_ID = run_id(RUN_PROFILE, 'full_run_id')
CONTEXT_RUN_ID = run_id(RUN_PROFILE, 'context_run_id')
SEQUENCE_RUN_ID = run_id(RUN_PROFILE, 'sequence_run_id')
SEQUENCE_TCN_RUN_ID = run_id(RUN_PROFILE, 'sequence_tcn_run_id')
VIDEO_LIGHTWEIGHT_RUN_ID = run_id(RUN_PROFILE, 'video_lightweight_run_id')
VIDEO_FROZEN_RUN_ID = run_id(RUN_PROFILE, 'video_frozen_run_id')
VIDEO_FINETUNE_RUN_ID = run_id(RUN_PROFILE, 'video_finetune_run_id')
PLAYER_SEASON_RUN_ID = run_id(RUN_PROFILE, 'player_season_run_id')
VLM_RUN_ID = run_id(RUN_PROFILE, 'vlm_run_id')
FUSION_RUN_ID = run_id(RUN_PROFILE, 'fusion_run_id')
METHOD_EVALUATION_REPORT_ID = run_id(RUN_PROFILE, 'method_evaluation_report_id')
VIDEO_ABLATION_REPORT_ID = run_id(RUN_PROFILE, 'video_ablation_report_id')
STRUCTURED_SEQUENCE_FEATURE_ID = artifact_id(RUN_PROFILE, 'structured_sequence_feature_id')
CLIP_EMBEDDING_FEATURE_ID = artifact_id(RUN_PROFILE, 'clip_embedding_feature_id')
PLAYER_SEASON_EMBEDDING_FEATURE_ID = artifact_id(RUN_PROFILE, 'player_season_embedding_feature_id')
VIDEO_LIGHTWEIGHT_FEATURE_ID = artifact_id(RUN_PROFILE, 'video_lightweight_feature_id', 'video_lightweight_features_mlb_2024_2026_v2')
VIDEO_EMBEDDING_FEATURE_ID = artifact_id(RUN_PROFILE, 'video_embedding_feature_id')
VLM_FEATURE_ID = artifact_id(RUN_PROFILE, 'vlm_feature_id')

print('RUN_PROFILE =', RUN_PROFILE_NAME)
print('RUN_PROFILE_PATH =', RUN_PROFILE_PATH)
print('DRIVE_REPO_DIR =', DRIVE_REPO_DIR)
print('LOCAL_REPO_DIR =', REPO_DIR)
print('BASE_DIR =', BASE_DIR)
print('NOTEBOOK_DIR =', NOTEBOOK_DIR)
print('src_dir =', src_dir)


In [ ]:
COMPACT_RUN_NAME = '35_cpu_evaluation_fusion_reports'
LOGGER = CompactRunLogger(BASE_DIR, COMPACT_RUN_NAME, run_profile_name=RUN_PROFILE_NAME)
FORCE_RERUN_ALL = False

VLM_PREDICTIONS_PATH = BASE_DIR / 'predictions' / VLM_RUN_ID / 'predictions_v1.parquet'
FUSION_SUMMARY_PATH = BASE_DIR / f'reports/preflight/full_fusion_{FUSION_RUN_ID}.json'
RESEARCH_SUMMARY_PATH = BASE_DIR / 'reports' / 'research_outputs' / FUSION_RUN_ID / 'summary.json'
METHOD_EVALUATION_SUMMARY_PATH = BASE_DIR / 'reports' / 'method_evaluation' / METHOD_EVALUATION_REPORT_ID / 'summary.json'
VIDEO_ABLATION_SUMMARY_PATH = BASE_DIR / 'reports' / 'ablation_compare' / VIDEO_ABLATION_REPORT_ID / 'summary.json'
EXPECTED_BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')

def _safe_read_json(path):
    try:
        return json.loads(path.read_text(encoding='utf-8')) if path.exists() else {}
    except Exception as exc:
        print('WARNING: 35 summary check failed for', path, exc)
        return {}

def _version_ok(value):
    return '_v2' in str(value or '') and '_v1' not in str(value or '')

def _assert_v2_runtime_contract():
    run_ids = RUN_PROFILE.get('run_ids', {})
    namespace = RUN_PROFILE.get('artifact_namespace', {})
    fusion_sources = RUN_PROFILE.get('execution', {}).get('fusion', {}).get('source_runs', [])
    problems = []
    if BASE_DIR != EXPECTED_BASE_DIR:
        problems.append(f'BASE_DIR must be {EXPECTED_BASE_DIR}, got {BASE_DIR}')
    if RUN_PROFILE_NAME != 'mlb_2024_2026_real_colab_v2.json':
        problems.append(f'RUN_PROFILE_NAME must be mlb_2024_2026_real_colab_v2.json, got {RUN_PROFILE_NAME}')
    for key, value in run_ids.items():
        if value and not _version_ok(value):
            problems.append(f'run_ids.{key} is not v2-isolated: {value}')
    for key, value in namespace.items():
        if value and not _version_ok(value):
            problems.append(f'artifact_namespace.{key} is not v2-isolated: {value}')
    for value in fusion_sources:
        if value and not _version_ok(value):
            problems.append(f'fusion source_run is not v2-isolated: {value}')
    if VLM_RUN_ID not in fusion_sources:
        problems.append(f'fusion source_runs does not include VLM run: {VLM_RUN_ID}')
    if f'{VLM_RUN_ID}_player_season_projection' not in fusion_sources:
        problems.append(f'fusion source_runs does not include VLM player-season projection: {VLM_RUN_ID}_player_season_projection')
    if problems:
        raise RuntimeError('35 v2 runtime contract failed:\n- ' + '\n- '.join(problems))

_assert_v2_runtime_contract()

fusion_summary = _safe_read_json(FUSION_SUMMARY_PATH)
fusion_has_vlm = any(
    row.get('run_id') == VLM_RUN_ID and bool(row.get('exists'))
    for row in fusion_summary.get('source_status', [])
    if isinstance(row, dict)
)
FORCE_FUSION_FOR_VLM = VLM_PREDICTIONS_PATH.exists() and not fusion_has_vlm
research_summary = _safe_read_json(RESEARCH_SUMMARY_PATH)
research_provenance = research_summary.get('provenance_checks', {}) if isinstance(research_summary, dict) else {}
research_has_v2_provenance = (
    research_summary.get('base_dir') == str(BASE_DIR)
    and research_summary.get('final_run_id') == FUSION_RUN_ID
    and research_summary.get('full_run_id') == FULL_RUN_ID
    and research_provenance.get('overall_status') == 'ok'
)
FORCE_RESEARCH_OUTPUTS_FOR_PROVENANCE = not research_has_v2_provenance

method_eval_summary = _safe_read_json(METHOD_EVALUATION_SUMMARY_PATH)
method_map = method_eval_summary.get('method_map', []) if isinstance(method_eval_summary, dict) else []
method_keys = {row.get('method_key') for row in method_map if isinstance(row, dict)}
method_run_ids = [str(row.get('run_id')) for row in method_map if isinstance(row, dict) and row.get('run_id')]
required_method_keys = {'context', 'pose_object_tcn', 'raw_video_lightweight', 'raw_video_frozen', 'raw_video_finetune', 'player_season', 'vlm', 'fusion'}
method_eval_has_required_v2 = (
    required_method_keys.issubset(method_keys)
    and bool(method_run_ids)
    and all(_version_ok(value) for value in method_run_ids)
    and bool(method_eval_summary.get('sample_counts'))
    and bool(method_eval_summary.get('method_metrics'))
    and bool(method_eval_summary.get('same_sample_intersection_metrics'))
)
FORCE_METHOD_EVALUATION_FOR_RESEARCH = not method_eval_has_required_v2

ablation_summary = _safe_read_json(VIDEO_ABLATION_SUMMARY_PATH)
ablation_run_ids = [str(row.get('run_id')) for row in ablation_summary.get('run_specs', []) if isinstance(row, dict) and row.get('run_id')]
ablation_has_v2_questions = (
    bool(ablation_run_ids)
    and all(_version_ok(value) for value in ablation_run_ids)
    and bool(ablation_summary.get('direct_questions') or ablation_summary.get('direct_questions_by_split'))
)
FORCE_VIDEO_ABLATION_FOR_V2 = not ablation_has_v2_questions
print('vlm_predictions_exists =', VLM_PREDICTIONS_PATH.exists())
print('fusion_has_vlm =', fusion_has_vlm)
print('force_fusion_for_vlm =', FORCE_FUSION_FOR_VLM)
print('research_has_v2_provenance =', research_has_v2_provenance)
print('force_research_outputs_for_provenance =', FORCE_RESEARCH_OUTPUTS_FOR_PROVENANCE)
print('method_eval_has_required_v2 =', method_eval_has_required_v2, 'force_method_evaluation_for_research =', FORCE_METHOD_EVALUATION_FOR_RESEARCH)
print('ablation_has_v2_questions =', ablation_has_v2_questions, 'force_video_ablation_for_v2 =', FORCE_VIDEO_ABLATION_FOR_V2)

RUN_PLAYER_SEASON_AGGREGATE = True
RUN_EVENT_METHOD_PLAYER_SEASON_PROJECTION_BEFORE_FUSION = True
RUN_FUSION = True
RUN_EVENT_METHOD_PLAYER_SEASON_PROJECTION_AFTER_FUSION = True
RUN_VIDEO_ABLATION_COMPARE = True
RUN_METHOD_EVALUATION = True
RUN_RESEARCH_OUTPUTS = True
RUN_STATIC_REPORT_BUILDER = True

# 35 は CPU/report wrapper なので、20 の ablation stage から重い raw-video fine-tune は起動しません。
# raw-video fine-tune は 33_gpu_sequence_and_video_models.ipynb 側で完了させます。
os.environ.setdefault('RUN_RAW_VIDEO_FINETUNE_IF_MISSING', '0')

stages = [
    {
        'notebook': '23_player_season_aggregate_baseline.ipynb',
        'label': 'Player-season mechanics prior BA/OPS/OBP/SLG baseline',
        'enabled': RUN_PLAYER_SEASON_AGGREGATE,
        'force': FORCE_RERUN_ALL,
        'expected_outputs': [f'predictions/{PLAYER_SEASON_RUN_ID}/predictions_v1.parquet', f'predictions/{PLAYER_SEASON_RUN_ID}/metrics_v1.json', f'reports/preflight/player_season_aggregate_{PLAYER_SEASON_RUN_ID}.json', f'datasets/player_season_targets/{PLAYER_SEASON_RUN_ID}/manifest.parquet'],
        'progress_files': [f'reports/preflight/player_season_aggregate_{PLAYER_SEASON_RUN_ID}.json'],
    },
    {
        'notebook': '26_event_method_player_season_projection.ipynb',
        'label': 'Project event methods to player-season targets before fusion',
        'enabled': RUN_EVENT_METHOD_PLAYER_SEASON_PROJECTION_BEFORE_FUSION,
        'force': True,
        'expected_outputs': ['reports/preflight/event_method_player_season_projection_summary_v1.json'],
        'progress_files': ['reports/preflight/event_method_player_season_projection_summary_v1.json'],
    },
    {
        'notebook': '15_full_fusion.ipynb',
        'label': 'Late fusion for event and player-season predictions',
        'enabled': RUN_FUSION,
        'force': FORCE_RERUN_ALL or FORCE_FUSION_FOR_VLM,
        'expected_outputs': [f'predictions/{FUSION_RUN_ID}/predictions_v1.parquet', f'predictions/{FUSION_RUN_ID}/metrics_v1.json', f'predictions/{FUSION_RUN_ID}/fusion_input_audit_v1.parquet', f'reports/preflight/full_fusion_{FUSION_RUN_ID}.json'],
        'progress_files': [f'reports/preflight/full_fusion_{FUSION_RUN_ID}.json'],
    },
    {
        'notebook': '26_event_method_player_season_projection.ipynb',
        'label': 'Project fusion event predictions to player-season targets',
        'enabled': RUN_EVENT_METHOD_PLAYER_SEASON_PROJECTION_AFTER_FUSION,
        'force': True,
        'expected_outputs': [f'predictions/{FUSION_RUN_ID}_player_season_projection/metrics_v1.json'],
        'progress_files': [f'reports/preflight/player_season_projection_{FUSION_RUN_ID}_player_season_projection.json'],
    },
    {
        'notebook': '20_video_ablation_compare.ipynb',
        'label': 'Video/method ablation comparison',
        'enabled': RUN_VIDEO_ABLATION_COMPARE,
        'force': FORCE_RERUN_ALL or FORCE_FUSION_FOR_VLM or FORCE_VIDEO_ABLATION_FOR_V2,
        'expected_outputs': [f'reports/ablation_compare/{VIDEO_ABLATION_REPORT_ID}/index.html', f'reports/ablation_compare/{VIDEO_ABLATION_REPORT_ID}/summary.json'],
        'progress_files': [f'reports/ablation_compare/{VIDEO_ABLATION_REPORT_ID}/summary.json'],
    },
    {
        'notebook': '25_method_evaluation.ipynb',
        'label': 'Per-method and fusion evaluation report',
        'enabled': RUN_METHOD_EVALUATION,
        'force': FORCE_RERUN_ALL or FORCE_FUSION_FOR_VLM or FORCE_METHOD_EVALUATION_FOR_RESEARCH,
        'expected_outputs': [f'reports/method_evaluation/{METHOD_EVALUATION_REPORT_ID}/index.html', f'reports/method_evaluation/{METHOD_EVALUATION_REPORT_ID}/summary.json', f'reports/method_evaluation/{METHOD_EVALUATION_REPORT_ID}/tables/method_metrics.csv', f'reports/method_evaluation/{METHOD_EVALUATION_REPORT_ID}/tables/sample_counts.csv', f'reports/method_evaluation/{METHOD_EVALUATION_REPORT_ID}/tables/same_sample_intersection_metrics.csv'],
        'progress_files': [f'reports/method_evaluation/{METHOD_EVALUATION_REPORT_ID}/summary.json'],
    },
    {
        'notebook': '22_research_outputs.ipynb',
        'label': 'Paper-style figures/tables/visual evidence',
        'enabled': RUN_RESEARCH_OUTPUTS,
        'force': FORCE_RERUN_ALL or FORCE_FUSION_FOR_VLM or FORCE_RESEARCH_OUTPUTS_FOR_PROVENANCE or FORCE_METHOD_EVALUATION_FOR_RESEARCH or FORCE_VIDEO_ABLATION_FOR_V2,
        'expected_outputs': [
            f'reports/research_outputs/{FUSION_RUN_ID}/index.html',
            f'reports/research_outputs/{FUSION_RUN_ID}/summary.json',
            f'reports/research_outputs/{FUSION_RUN_ID}/tables/model_design.csv',
            f'reports/research_outputs/{FUSION_RUN_ID}/tables/provenance_checks.csv',
            f'reports/research_outputs/{FUSION_RUN_ID}/tables/method_evaluation_sample_counts.csv',
            f'reports/research_outputs/{FUSION_RUN_ID}/tables/method_baseline_comparison.csv',
            f'reports/research_outputs/{FUSION_RUN_ID}/figures/method_available_rows_by_target.png',
            f'reports/research_outputs/{FUSION_RUN_ID}/figures/method_primary_metric_matrix.png',
            f'reports/research_outputs/{FUSION_RUN_ID}/figures/method_delta_vs_context.png',
        ],
        'progress_files': [f'reports/research_outputs/{FUSION_RUN_ID}/summary.json'],
    },
    {
        'notebook': '09_report_builder.ipynb',
        'label': 'Static browser reports',
        'enabled': RUN_STATIC_REPORT_BUILDER,
        'force': FORCE_RERUN_ALL or FORCE_FUSION_FOR_VLM or FORCE_RESEARCH_OUTPUTS_FOR_PROVENANCE or FORCE_METHOD_EVALUATION_FOR_RESEARCH or FORCE_VIDEO_ABLATION_FOR_V2,
        'expected_outputs': [f'reports/target_availability/{FUSION_RUN_ID}/index.html'],
        'progress_files': [],
    },
]
LOGGER.run_stages(ipython=get_ipython(), notebook_dir=NOTEBOOK_DIR, stages=stages)


In [ ]:

def _assert_final_research_outputs_v2():
    summary = _safe_read_json(RESEARCH_SUMMARY_PATH)
    report_root = BASE_DIR / 'reports' / 'research_outputs' / FUSION_RUN_ID
    tables_dir = report_root / 'tables'
    figures_dir = report_root / 'figures'
    videos_dir = report_root / 'videos'
    method_comparison = summary.get('method_comparison', {}) if isinstance(summary, dict) else {}
    provenance = summary.get('provenance_checks', {}) if isinstance(summary, dict) else {}
    problems = []
    if not RESEARCH_SUMMARY_PATH.exists():
        problems.append(f'missing research summary: {RESEARCH_SUMMARY_PATH}')
    if summary.get('base_dir') != str(BASE_DIR):
        problems.append(f'research summary base_dir mismatch: {summary.get("base_dir")}')
    if summary.get('final_run_id') != FUSION_RUN_ID or not _version_ok(summary.get('final_run_id')):
        problems.append(f'research summary final_run_id is not active v2 fusion: {summary.get("final_run_id")}')
    if summary.get('full_run_id') != FULL_RUN_ID or not _version_ok(summary.get('full_run_id')):
        problems.append(f'research summary full_run_id is not active v2 full run: {summary.get("full_run_id")}')
    if provenance.get('overall_status') != 'ok':
        problems.append(f'provenance overall_status is not ok: {provenance.get("overall_status")}')
    for check in provenance.get('checks', []):
        if isinstance(check, dict) and check.get('status') != 'ok':
            problems.append(f'provenance warning: {check}')
    required_tables = [
        'model_design.csv',
        'provenance_checks.csv',
        'fusion_source_status.csv',
        'method_evaluation_method_map.csv',
        'method_evaluation_sample_counts.csv',
        'method_evaluation_method_metrics.csv',
        'method_evaluation_same_sample_intersection_metrics.csv',
        'method_evaluation_best_by_target.csv',
        'method_baseline_comparison.csv',
        'metrics_by_run_target.csv',
        'fusion_input_audit_summary.csv',
    ]
    for name in required_tables:
        path = tables_dir / name
        if not path.exists() or path.stat().st_size == 0:
            problems.append(f'missing/empty research table: {path}')
    required_figures = [
        'method_available_rows_by_target.png',
        'method_primary_metric_matrix.png',
        'method_delta_vs_context.png',
        'model_metrics_mae.png',
        'target_availability.png',
        'fusion_input_provenance.png',
        'artifact_scale.png',
        'prediction_scatter_ev_la.png',
        'residual_distribution_ev_la.png',
        'contact_frame_sheet.jpg',
    ]
    for name in required_figures:
        path = figures_dir / name
        if not path.exists() or path.stat().st_size == 0:
            problems.append(f'missing/empty research figure: {path}')
    overlay_videos = list(videos_dir.glob('**/*.mp4')) if videos_dir.exists() else []
    if not overlay_videos:
        problems.append(f'no research overlay mp4 found under {videos_dir}')
    if not method_comparison.get('sample_count_rows'):
        problems.append('method_comparison.sample_count_rows is empty')
    if not method_comparison.get('method_metrics_rows'):
        problems.append('method_comparison.method_metrics_rows is empty')
    if not method_comparison.get('baseline_comparison_rows'):
        problems.append('method_comparison.baseline_comparison_rows is empty')
    if problems:
        raise RuntimeError('35 final v2 research output verification failed:\n- ' + '\n- '.join(problems))
    print('35 final v2 research output verification = ok')
    print('research_summary =', RESEARCH_SUMMARY_PATH)
    print('research_tables =', tables_dir)
    print('research_figures =', figures_dir)
    print('research_overlay_videos =', len(overlay_videos))
    print('method_comparison_rows =', {
        'sample_count_rows': method_comparison.get('sample_count_rows'),
        'method_metrics_rows': method_comparison.get('method_metrics_rows'),
        'same_sample_metric_rows': method_comparison.get('same_sample_metric_rows'),
        'baseline_comparison_rows': method_comparison.get('baseline_comparison_rows'),
    })

_assert_final_research_outputs_v2()
